# 02 · Bootstrap Confidence Intervals

## What this notebook covers
Point estimates (AUC = 0.83) conceal uncertainty. Bootstrap resampling turns a single held-out set into a full sampling distribution — no distributional assumptions required.

## Why it matters
Regulatory submissions, A/B decision gates, and fairness audits all need intervals, not point estimates. Bootstrap CIs are the modern standard because they work for any metric (AUC, F1, NDCG) without requiring the metric to be normally distributed.

## Two flavours
| Method | When to use |
|--------|------------|
| **Percentile** | Large samples (n > 200), symmetric metrics |
| **BCa** (bias-corrected and accelerated) | Small samples, skewed metrics, when you want coverage guarantees |

BCa corrects for two biases: systematic offset between bootstrap and true distribution (bias correction) and non-constant standard error across the parameter space (acceleration). It's the default in SciPy's `bootstrap` as of 1.7.

In [ ]:
import numpy as np
import pandas as pd
from sklearn.datasets import make_classification
from sklearn.ensemble import GradientBoostingClassifier, RandomForestClassifier
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from scipy import stats
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore")
np.random.seed(42)

# ── Shared dataset ──────────────────────────────────────────────────────────────
X, y = make_classification(n_samples=1000, n_features=20, n_informative=10, random_state=0)
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.3, random_state=0)

clf = GradientBoostingClassifier(n_estimators=100, random_state=0)
clf.fit(X_tr, y_tr)
proba = clf.predict_proba(X_te)[:, 1]
point_auc = roc_auc_score(y_te, proba)
print(f"Point AUC: {point_auc:.4f}")

In [ ]:
def bootstrap_ci(y_true, y_score, n_bootstrap=2000, alpha=0.05, method="bca"):
    """
    Bootstrap CI for AUC.
    method: 'percentile' | 'bca'
    Returns (lower, upper).
    """
    n = len(y_true)
    boot_aucs = []
    for _ in range(n_bootstrap):
        idx = np.random.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        boot_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))

    boot_aucs = np.array(boot_aucs)

    if method == "percentile":
        lo = np.percentile(boot_aucs, 100 * alpha / 2)
        hi = np.percentile(boot_aucs, 100 * (1 - alpha / 2))
        return lo, hi

    # BCa
    # Bias correction
    z0 = stats.norm.ppf((boot_aucs < point_auc).mean())
    # Acceleration (jackknife)
    jack_aucs = []
    for i in range(n):
        idx = np.concatenate([np.arange(i), np.arange(i+1, n)])
        if len(np.unique(y_true[idx])) < 2:
            continue
        jack_aucs.append(roc_auc_score(y_true[idx], y_score[idx]))
    jack_aucs = np.array(jack_aucs)
    jack_mean = jack_aucs.mean()
    num = ((jack_mean - jack_aucs) ** 3).sum()
    den = 6 * ((jack_mean - jack_aucs) ** 2).sum() ** 1.5
    a = num / den if den != 0 else 0

    z_alpha  = stats.norm.ppf(alpha / 2)
    z_1alpha = stats.norm.ppf(1 - alpha / 2)
    p_lo = stats.norm.cdf(z0 + (z0 + z_alpha)  / (1 - a * (z0 + z_alpha)))
    p_hi = stats.norm.cdf(z0 + (z0 + z_1alpha) / (1 - a * (z0 + z_1alpha)))
    lo = np.percentile(boot_aucs, 100 * p_lo)
    hi = np.percentile(boot_aucs, 100 * p_hi)
    return lo, hi

lo_pct, hi_pct = bootstrap_ci(y_te, proba, method="percentile")
lo_bca, hi_bca = bootstrap_ci(y_te, proba, method="bca")
print(f"Percentile 95% CI: [{lo_pct:.4f}, {hi_pct:.4f}]")
print(f"BCa        95% CI: [{lo_bca:.4f}, {hi_bca:.4f}]")

## Paired bootstrap for model comparison
Instead of two independent CIs, compute a CI on the *difference* directly. This accounts for the fact that both models are evaluated on the same test set (paired observations).

In [ ]:
def paired_bootstrap_ci(y_true, score_a, score_b, n_bootstrap=2000, alpha=0.05):
    """95% CI on (AUC_A - AUC_B) using paired bootstrap."""
    n = len(y_true)
    deltas = []
    for _ in range(n_bootstrap):
        idx = np.random.randint(0, n, n)
        if len(np.unique(y_true[idx])) < 2:
            continue
        auc_a = roc_auc_score(y_true[idx], score_a[idx])
        auc_b = roc_auc_score(y_true[idx], score_b[idx])
        deltas.append(auc_a - auc_b)
    deltas = np.array(deltas)
    lo = np.percentile(deltas, 100 * alpha / 2)
    hi = np.percentile(deltas, 100 * (1 - alpha / 2))
    return lo, hi, deltas

# Compare GBM vs RF
clf_rf = RandomForestClassifier(n_estimators=100, random_state=0)
clf_rf.fit(X_tr, y_tr)
proba_rf = clf_rf.predict_proba(X_te)[:, 1]

auc_gbm = roc_auc_score(y_te, proba)
auc_rf  = roc_auc_score(y_te, proba_rf)

lo_d, hi_d, deltas = paired_bootstrap_ci(y_te, proba, proba_rf)
print(f"GBM AUC: {auc_gbm:.4f}  |  RF AUC: {auc_rf:.4f}")
print(f"Delta (GBM - RF): {auc_gbm - auc_rf:.4f}")
print(f"Paired bootstrap 95% CI on delta: [{lo_d:.4f}, {hi_d:.4f}]")
print(f"Zero in CI? {'Yes → not significant' if lo_d <= 0 <= hi_d else 'No → significant difference'}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4))

# Bootstrap AUC distribution
n_boot = 2000
boot_aucs_plot = []
for _ in range(n_boot):
    idx = np.random.randint(0, len(y_te), len(y_te))
    if len(np.unique(y_te[idx])) < 2: continue
    boot_aucs_plot.append(roc_auc_score(y_te[idx], proba[idx]))
boot_aucs_plot = np.array(boot_aucs_plot)

axes[0].hist(boot_aucs_plot, bins=50, color="steelblue", alpha=0.8, edgecolor="white")
axes[0].axvline(point_auc, color="red", lw=2, label=f"Point AUC {point_auc:.3f}")
axes[0].axvline(lo_bca, color="orange", lw=1.5, linestyle="--", label=f"BCa [{lo_bca:.3f}, {hi_bca:.3f}]")
axes[0].axvline(hi_bca, color="orange", lw=1.5, linestyle="--")
axes[0].set_title("Bootstrap AUC distribution (GBM)")
axes[0].legend(fontsize=9)
axes[0].set_xlabel("AUC")

# Delta distribution
axes[1].hist(deltas, bins=50, color="seagreen", alpha=0.8, edgecolor="white")
axes[1].axvline(0, color="black", lw=1.5, linestyle=":", label="No difference")
axes[1].axvline(lo_d, color="orange", lw=1.5, linestyle="--", label=f"95% CI")
axes[1].axvline(hi_d, color="orange", lw=1.5, linestyle="--")
axes[1].set_title("Paired bootstrap: GBM − RF delta")
axes[1].legend(fontsize=9)
axes[1].set_xlabel("ΔAUC")

plt.tight_layout()
plt.savefig(f"{base}/02_bootstrap_ci.png", dpi=120)
plt.show()
print("Saved 02_bootstrap_ci.png")